# 10 - AgentCore Identity
Create workload identities for agents, configure OAuth credential providers, and test token exchange for secure service-to-service communication.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
ACCOUNT_ID = config["ACCOUNT_ID"]

tracker = ResourceTracker()
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Create Workload Identity

A workload identity represents an agent's identity for accessing external services.

In [ ]:
IDENTITY_NAME = f"{PREFIX}-agent-identity"

# Check if identity exists
existing_identities = agentcore.list_workload_identities()
existing = next(
    (i for i in existing_identities.get("identities", []) if i["name"] == IDENTITY_NAME),
    None
)

if existing:
    identity_id = existing["id"]
    print(f"\u2713 Workload identity already exists: {identity_id}")
else:
    print(f"\u231b Creating workload identity: {IDENTITY_NAME}...")
    response = agentcore.create_workload_identity(
        name=IDENTITY_NAME,
        description="OrgIQ agent workload identity for secure service access",
        allowedAgents=[
            f"{PREFIX}-supervisor",
            f"{PREFIX}-search",
            f"{PREFIX}-scoring"
        ],
        tags={"project": PREFIX}
    )
    identity_id = response["id"]
    print(f"\u2713 Workload identity created: {identity_id}")

## Step 3: Create OAuth Credential Provider

Configure an OAuth 2.0 credential provider for external API access.

In [ ]:
CREDENTIAL_NAME = f"{PREFIX}-oauth-provider"

# Check if credential provider exists
existing_creds = agentcore.list_credential_providers(
    workloadIdentityId=identity_id
)
existing_cred = next(
    (c for c in existing_creds.get("credentialProviders", []) if c["name"] == CREDENTIAL_NAME),
    None
)

if existing_cred:
    credential_id = existing_cred["id"]
    print(f"\u2713 Credential provider already exists: {credential_id}")
else:
    print(f"\u231b Creating OAuth credential provider: {CREDENTIAL_NAME}...")
    response = agentcore.create_credential_provider(
        workloadIdentityId=identity_id,
        name=CREDENTIAL_NAME,
        description="OAuth 2.0 provider for external service integration",
        credentialProviderType="OAUTH2",
        oAuth2Configuration={
            "grantType": "CLIENT_CREDENTIALS",
            "tokenEndpoint": "https://auth.example.com/oauth2/token",
            "scopes": ["read:org", "read:docs"],
            "clientId": f"{PREFIX}-agent-client",
            "clientSecretArn": f"arn:aws:secretsmanager:{REGION}:{ACCOUNT_ID}:secret:{PREFIX}-oauth-secret"
        }
    )
    credential_id = response["id"]
    print(f"\u2713 OAuth credential provider created: {credential_id}")

## Step 4: Create API Key Credential Provider

Some services use API keys. Create a provider for those.

In [ ]:
API_KEY_CRED_NAME = f"{PREFIX}-apikey-provider"

existing_apikey = next(
    (c for c in existing_creds.get("credentialProviders", []) if c["name"] == API_KEY_CRED_NAME),
    None
)

if existing_apikey:
    apikey_cred_id = existing_apikey["id"]
    print(f"\u2713 API Key provider already exists: {apikey_cred_id}")
else:
    print(f"\u231b Creating API Key credential provider: {API_KEY_CRED_NAME}...")
    response = agentcore.create_credential_provider(
        workloadIdentityId=identity_id,
        name=API_KEY_CRED_NAME,
        description="API Key provider for external integrations",
        credentialProviderType="API_KEY",
        apiKeyConfiguration={
            "secretArn": f"arn:aws:secretsmanager:{REGION}:{ACCOUNT_ID}:secret:{PREFIX}-api-key",
            "headerName": "X-API-Key"
        }
    )
    apikey_cred_id = response["id"]
    print(f"\u2713 API Key credential provider created: {apikey_cred_id}")

## Step 5: Get Workload Identity Details

In [ ]:
identity_info = agentcore.get_workload_identity(
    workloadIdentityId=identity_id
)

print(f"Workload Identity: {identity_info['name']}")
print(f"  ID: {identity_info['id']}")
print(f"  Status: {identity_info.get('status', 'N/A')}")
print(f"  Allowed Agents: {identity_info.get('allowedAgents', [])}")
print(f"  Created: {identity_info.get('createdAt', 'N/A')}")

## Step 6: List Credential Providers

In [ ]:
cred_providers = agentcore.list_credential_providers(
    workloadIdentityId=identity_id
)

print(f"Credential providers ({len(cred_providers.get('credentialProviders', []))}):\n")
for cred in cred_providers.get("credentialProviders", []):
    print(f"  {cred['name']}")
    print(f"    Type: {cred.get('credentialProviderType', 'N/A')}")
    print(f"    ID: {cred['id']}")
    print()

## Step 7: Test Token Exchange

Request a token using the workload identity (simulating what an agent does at runtime).

In [ ]:
# Request token exchange for the agent identity
# Note: This requires the actual OAuth endpoint to be reachable
try:
    token_response = agentcore.get_credential(
        workloadIdentityId=identity_id,
        credentialProviderId=credential_id,
        agentId=f"{PREFIX}-search"
    )
    
    print("\u2713 Token exchange successful")
    print(f"  Token type: {token_response.get('tokenType', 'N/A')}")
    print(f"  Expires in: {token_response.get('expiresIn', 'N/A')}s")
    # Don't print the actual token for security
    token_preview = token_response.get('accessToken', '')[:20]
    print(f"  Token preview: {token_preview}...")
except Exception as e:
    print(f"\u231b Token exchange not available (expected in notebook env): {type(e).__name__}")
    print(f"  This works at runtime when agents are deployed to AgentCore.")
    print(f"  The agent would call: agentcore.get_credential(...)")

## Step 8: Identity in Agent Code

Show how an agent uses workload identity at runtime.

In [ ]:
# Example: How an agent uses workload identity in its tool code
from strands import tool

@tool
def call_external_api(endpoint: str, query: str) -> str:
    """Call an external API using the agent's workload identity credentials.
    
    Args:
        endpoint: The API endpoint to call
        query: The query parameter to send
    """
    import requests
    
    # Get credential from AgentCore identity service
    agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)
    cred = agentcore_client.get_credential(
        workloadIdentityId=identity_id,
        credentialProviderId=credential_id,
        agentId=f"{PREFIX}-search"
    )
    
    # Use the token to authenticate
    headers = {
        "Authorization": f"Bearer {cred['accessToken']}",
        "Content-Type": "application/json"
    }
    
    response = requests.get(endpoint, headers=headers, params={"q": query})
    return response.json()

print("\u2713 External API tool with identity example defined")
print("  At runtime, this tool would:")
print("  1. Request credentials from AgentCore Identity")
print("  2. Use the token to authenticate with external service")
print("  3. AgentCore handles token refresh automatically")

## Step 9: Identity Integration with Gateway

In [ ]:
# Show how identity integrates with the MCP Gateway
GATEWAY_ID = config["GATEWAY_ID"]

# When a gateway target requires authentication, the identity is used automatically
print("Identity + Gateway integration:")
print(f"  Gateway: {GATEWAY_ID}")
print(f"  Identity: {identity_id}")
print()
print("  Flow:")
print("  1. Agent calls gateway tool")
print("  2. Gateway checks policy engine (Cedar)")
print("  3. If allowed, gateway retrieves credential via identity")
print("  4. Gateway invokes target with authenticated request")
print("  5. Response returned to agent")
print()
print("\u2713 Identity-Gateway integration documented")

## Step 10: Save Configuration

In [ ]:
config["WORKLOAD_IDENTITY_NAME"] = IDENTITY_NAME
Path("config.json").write_text(json.dumps(config, indent=2))

print(f"\u2713 Config saved")
print(f"  WORKLOAD_IDENTITY_NAME: {IDENTITY_NAME}")

## Summary

- [x] Workload identity created for agent fleet
- [x] OAuth 2.0 credential provider configured (client_credentials grant)
- [x] API Key credential provider configured
- [x] Token exchange flow demonstrated
- [x] Agent tool code example with identity integration
- [x] Gateway + Identity integration flow documented
- [x] Multi-agent identity sharing (supervisor, search, scoring)

**Next:** [11-ag-ui-streaming.ipynb](11-ag-ui-streaming.ipynb) — Implement AG-UI protocol for frontend streaming